# Unlabeled Samples Predictions

Given a trained checkpoint path, this notebook predicts classes for unlabeled plaque samples and visualizes up to `num_examples` predictions per class.

It scans at most `max_samples` unlabeled rows.

In [7]:
import sys
sys.path.append('/workspace')
sys.path.append('/workspace/src')
import os
# change the current working directory to the workspace
os.chdir('/workspace')
from tqdm import tqdm
import pandas as pd

from src.models.config import Config
from src.supervised_runner import SupervisedRunner
from src.semi_supervised_runner import SemiSupervisedRunner
from src.self_supervised_runner import SelfSupervisedRunner

In [10]:
def load_model(train_type, run_folder_path, load_dataloaders=False):
    if not os.path.exists(os.path.join(run_folder_path, 'config.txt')):
        raise FileNotFoundError(f"Config file not found at {os.path.join(run_folder_path, 'config.txt')}")
    config = Config.from_txt(os.path.join(run_folder_path, 'config.txt'))
    if train_type == 'supervised':
        runner = SupervisedRunner(config, 'single')
    elif train_type == 'semi_supervised':
        runner = SemiSupervisedRunner(config, 'single')
    elif train_type == 'self_supervised':
        runner = SelfSupervisedRunner(config, 'single')
    else:
        raise ValueError(f"Invalid train type: {train_type}")
    model = runner.load_model_from_checkpoint(os.path.join(run_folder_path, 'checkpoints', 'best_model_cv.ckpt'))

    if load_dataloaders:
        if runner._type() == 'supervised':
            labeled_dataloader,_,_= runner.load_dataloaders(runner.labeled_data_df, pd.DataFrame(), pd.DataFrame(), runner.unlabeled_data_df)
            return model, labeled_dataloader
        else:
            labeled_dataloader,_,_,unlabeled_dataloader= runner.load_dataloaders(runner.labeled_data_df, pd.DataFrame(), pd.DataFrame(), runner.unlabeled_data_df)
            return model, (labeled_dataloader, unlabeled_dataloader)
    else:
        return model

In [ ]:
TRAIN_TYPE = 'self_supervised'
TRAIN_METHOD = 'simclr'
FEATURE_EXTRACTOR = 'resnet18'
RUN_FOLDER_NAME = '20260420_185825'

run_folder_path = '/workspace/runs/cross_validate/' + TRAIN_TYPE + '/' + TRAIN_METHOD + '/' + FEATURE_EXTRACTOR + '/' + RUN_FOLDER_NAME
simclr_model, (labeled_dataloader, unlabeled_dataloader) = load_model(TRAIN_TYPE, run_folder_path, load_dataloaders=True)
print(simclr_model)

Seed set to 44


Random seeds set to 44 for reproducibility
Blocks to freeze: [Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False), BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64,

In [ ]:
TRAIN_TYPE = 'supervised'
TRAIN_METHOD = ''
FEATURE_EXTRACTOR = 'resnet18'
RUN_FOLDER_NAME = '20260420_195502'

run_folder_path = '/workspace/runs/cross_validate/' + TRAIN_TYPE + '/' + TRAIN_METHOD + '/' + FEATURE_EXTRACTOR + '/' + RUN_FOLDER_NAME
simclr_model, (labeled_dataloader, unlabeled_dataloader) = load_model(TRAIN_TYPE, run_folder_path, load_dataloaders=True)
print(simclr_model)

In [ ]:
if train_mode in ("self_supervised", "semi_supervised"):
    _labeled_dl, _val_dl, _test_dl, unlabeled_dataloader = runner.load_dataloaders(
        runner.labeled_data_df,
        pd.DataFrame(),
        pd.DataFrame(),
        runner.unlabeled_data_df,
    )
    unlabeled_dataset = unlabeled_dataloader.dataset
else:
    raise ValueError(
        "This notebook uses runner unlabeled dataloader flow, which is available for self/semi-supervised runners. "
        "Use a self_supervised or semi_supervised checkpoint path."
    )

# Collect examples per predicted class
label_ids = sorted(config.label_to_name.keys())
examples_by_label = {label_id: [] for label_id in label_ids}

with torch.no_grad():
    for idx in range(len(unlabeled_dataset)):
        image_path, _raw, transformed_stack, extra_features, _label = unlabeled_dataset[idx]
        if transformed_stack.numel() == 0:
            continue

        x_image = transformed_stack[0].unsqueeze(0).to(device)
        x_features = extra_features.unsqueeze(0).to(device) if extra_features.numel() > 0 else None

        pred = model.predict(x_image, x_features=x_features)
        pred_label = int(pred.item())

        if pred_label in examples_by_label and len(examples_by_label[pred_label]) < num_examples:
            examples_by_label[pred_label].append(str(image_path))

        if all(len(v) >= num_examples for v in examples_by_label.values()):
            break

# Plot grid: columns = predicted class, rows = examples
n_cols = len(label_ids)
n_rows = num_examples
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)
if n_cols == 1:
    axes = np.expand_dims(axes, axis=1)

for col_idx, label_id in enumerate(label_ids):
    class_name = config.label_to_name[label_id]
    class_examples = examples_by_label[label_id]

    for row_idx in range(n_rows):
        ax = axes[row_idx, col_idx]
        ax.axis("off")

        if row_idx < len(class_examples):
            img = plt.imread(class_examples[row_idx])
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "Not found", ha="center", va="center", fontsize=9)

        if row_idx == 0:
            ax.set_title(f"{class_name}\n(n={len(class_examples)})", fontsize=10)

plt.tight_layout()
plt.show()

print(f"Scanned up to {sample_n} unlabeled samples.")
for label_id in label_ids:
    print(f"{config.label_to_name[label_id]}: {len(examples_by_label[label_id])}/{num_examples}")

ValueError: This notebook uses runner unlabeled dataloader flow, which is available for self/semi-supervised runners. Use a self_supervised or semi_supervised checkpoint path.